# 🎯 From tracking data to a scored submission

This notebook takes you the whole way: load the tracking, build a possession label for every
frame, write the submission file, and score it. By the end you will have beaten the trivial
baselines, and you will have seen on your own screen why the challenge is ranked on `spellF1`
and not on per-frame accuracy.

It needs **numpy, pandas and matplotlib** — no kloppy, no video decoder. The algorithm is
deliberately the simplest thing that is not a constant guess, so everything you add from here
is an improvement you can measure.

---

## 1. Where the data is

Nothing in this repository hard-codes a filename. `games.asset_paths()` builds every path from
one place, so if you keep the bundle somewhere else you set `MTAH_DATA_DIR` and change nothing
else.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import games

for name, path in sorted(games.asset_paths(source='tactical').items()):
    print('%-14s %-46s %s' % (name, os.path.basename(path),
                              'ok' if os.path.isfile(path) else 'MISSING'))

The videos are large and this notebook does not need them. The two ATD files are what matter.

## 2. Read the tracking, and meet the problem

`starter/baseline.py` reads the raw ATD with the standard library and numpy — one pass over
199,319 lines, a minute or two.

For every frame it finds the **tracked player nearest the ball** and returns that team. Where
the tracking cannot see the ball, it returns nothing.

In [ ]:
import baseline

codes = baseline.build()   # 0 = home, 1 = away, -1 = no answer

### The ball is missing far more often than you would guess

This is the most important single fact about the dataset, and the reason a ball-only approach
is harder than it sounds. Look at where the gaps are before you design around them.

In [ ]:
seen = codes != baseline.NOBODY
print('ball located on %d of %d frames (%.1f%%)' % (seen.sum(), len(seen), 100 * seen.mean()))

# Coverage per minute, so the halftime hole and the in-play gaps are both visible.
per_minute = seen[:len(seen) // 1500 * 1500].reshape(-1, 1500).mean(axis=1)

fig, ax = plt.subplots(figsize=(13, 3))
ax.fill_between(np.arange(len(per_minute)), per_minute * 100, color='#1f77b4', alpha=.8)
ax.axhline(100 * seen.mean(), color='#d62728', ls='--', label='match average')
ax.set_xlabel('minute of video'); ax.set_ylabel('% frames with a ball')
ax.set_ylim(0, 100); ax.legend(); ax.set_title('Ball coverage in the tracking data')
plt.show()

The trough in the middle is halftime, where there is no ground truth either. The dips during
play are the ones that cost you: occlusions, crowded boxes, and the ball in the air.

## 3. Your first submission

A submission is a CSV with one row per video frame and a label in `H` / `A` / `D` / `X`. That
is the entire format. Write the raw nearest-player answer and check it.

In [ ]:
labels_raw = baseline.smooth(codes, 0)   # no smoothing yet
path = baseline.write_submission('../evaluation/results/my_first.csv', labels_raw)

print(pd.Series(labels_raw).value_counts().to_string())
print()
print(open(path).read(40))

In [ ]:
!python ../evaluation/validate_submission.py ../evaluation/results/my_first.csv

**Always run the validator on the exact file you are about to submit.** It applies the same
checks the official scorer applies, and a file that fails them scores nothing.

Read the last line as well as the first. The format is fine — but look at how many possessions
it claims.

## 4. Score it

Against the public five minutes of ground truth that ship with the repository.

In [ ]:
sys.path.insert(0, os.path.abspath('../evaluation'))
import possession_metrics as pm

gt = pm.label_codes(pm.read_label_csv('../evaluation/gt/BAR-ATH_possession_gt_public.csv'))

def score(labels):
    """The two headline numbers, plus how many possessions the labels imply."""
    r = pm.evaluate(gt, pm.label_codes(labels))
    return {'macroF1': round(r['macro_f1'], 4), 'spellF1': round(r['spell_f1_mean'], 4),
            'possessions': r['n_pred_spells'],
            'mean_dur_s': round(r['mean_spell_duration_s'], 1)}

print('ground truth: 17 possessions of mean 14.2 s')
print(score(labels_raw))

`macroF1` looks almost respectable. `spellF1` is close to zero.

The reason is in the possession count. The raw labels flicker between teams frame by frame, so
instead of a few dozen possessions they report hundreds of tiny ones. Per-frame scoring barely
notices. `spellF1` — which counts each possession once, the way a coach does — notices at once.

## 5. Smooth it, and watch the two metrics disagree

One change: a majority vote over a window centred on each frame. Nothing else about the
algorithm changes.

In [ ]:
rows = []
for window in (0, 12, 25, 50, 75, 125, 200):
    rows.append(dict(window=window, **score(baseline.smooth(codes, window))))

table = pd.DataFrame(rows).set_index('window')
table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(table.index, table['macroF1'], 'o-', label='macroF1  (per frame)')
ax.plot(table.index, table['spellF1'], 'o-', label='spellF1  (per possession, ranked on)')
ax.set_xlabel('smoothing half-window (frames)'); ax.set_ylabel('score')
ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=.3)
ax.set_title('One parameter. Two metrics. Very different opinions.')
plt.show()

`macroF1` is nearly flat across the whole sweep. `spellF1` climbs by a factor of five, then
falls again.

That gap is the argument of [EVALUATION.md](../EVALUATION.md): no per-frame metric can
represent possession, because the overwhelming majority of frames sit in the middle of one where
nothing is happening. Optimise for the wrong one and you will tune yourself into a worse
submission.

## 6. A warning that costs nothing to learn here

The window that wins above is **not** the window that wins on the full match.

Scored on the whole match, `±50` beats `±125` — the reverse of what these five minutes say.
Seventeen possessions cannot rank anything: one missed possession moves `spellF1` by roughly six
points.

Use the public window to check that your pipeline is **correct**, never to choose between two
versions of it. When a gap looks real, run `--bootstrap 1000` and see whether the intervals
overlap. They usually will.

## 7. Where to go from here

Everything below is untouched by this baseline:

| | |
|---|---|
| **Dead ball** | it says `D` only when the tracking lost the ball. Real stoppages have structure — players walking, the ball still. Start from the Smart Tagging set pieces. |
| **Possession structure** | nothing here knows a possession has a beginning and an end. Modelling intervals rather than frames is the obvious next step. |
| **Smart Tagging** | 638 tactical phases, each with a team tag. A free prior on both halves of the label. See `01_smart_tagging.ipynb`. |
| **The homography** | project tracking onto the video and look at your own errors. See `starter/load_homography.py`. |
| **The broadcast recording** | a second camera with its own tracking, on its own timeline. |

And when a score surprises you, open `evaluation/reports/<name>_transitions.csv`. Every
possession change is listed as `matched`, `missed` or `spurious`, with the video frame it
happened on — so you can go and watch what fooled your model.